In [1]:
import pandas as pd
from joblib import dump
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Load pre-match features
data = pd.read_csv(
    "data/processed/epl_features.csv",
    parse_dates=["Date"]
)

feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
]

# Use exactly the same time-based split as our Poisson baseline
train = data[data["Season"] != "2024-25"].copy()
test = data[data["Season"] == "2024-25"].copy()

X_train = train[feature_columns]
X_test = test[feature_columns]

# One model predicts home goals; the other predicts away goals
model_settings = {
    "objective": "count:poisson",
    "n_estimators": 400,
    "max_depth": 3,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "random_state": 42,
    "n_jobs": -1,
}

xgb_home_model = XGBRegressor(**model_settings)
xgb_away_model = XGBRegressor(**model_settings)

xgb_home_model.fit(X_train, train["FTHG"])
xgb_away_model.fit(X_train, train["FTAG"])

# Predict expected goals for the untouched 2024–25 test season
test["xgb_predicted_home_goals"] = xgb_home_model.predict(X_test)
test["xgb_predicted_away_goals"] = xgb_away_model.predict(X_test)

# Compare goal-prediction error with the Poisson baseline
xgb_home_mae = mean_absolute_error(
    test["FTHG"], test["xgb_predicted_home_goals"]
)
xgb_away_mae = mean_absolute_error(
    test["FTAG"], test["xgb_predicted_away_goals"]
)

print(f"XGBoost home-goal MAE: {xgb_home_mae:.3f}")
print(f"XGBoost away-goal MAE: {xgb_away_mae:.3f}")
print()
print("Poisson baseline home-goal MAE: 1.001")
print("Poisson baseline away-goal MAE: 0.893")

# Save the trained models for our future prediction interface
dump(xgb_home_model, "models/xgb_home_goals.joblib")
dump(xgb_away_model, "models/xgb_away_goals.joblib")

print("\nSaved both XGBoost models in the models folder.")

XGBoost home-goal MAE: 1.019
XGBoost away-goal MAE: 0.896

Poisson baseline home-goal MAE: 1.001
Poisson baseline away-goal MAE: 0.893

Saved both XGBoost models in the models folder.
